In [10]:
from bs4 import BeautifulSoup
import requests
import csv

In [11]:
link = "https://vnexpress.net/thoi-su"

In [12]:
'''
workflow:

1. send request
2. parse text to html 
3. get href attributes and save to a list/set
4. loop thru the list/set and send request again
5. extract data
'''


'\nworkflow:\n\n1. send request\n2. parse text to html \n3. get href attributes and save to a list/set\n4. loop thru the list/set and send request again\n5. extract data\n'

In [13]:
def send_requests(link):
    r = requests.get(link)
    
    if r.status_code == 200:
        html_text = BeautifulSoup(r.text, 'html.parser') # parse raw text into html 
    
    return r.status_code, html_text

In [14]:
def get_links(html_text): # get parsed html and return a set of href urls
    s = html_text.find_all('h3', class_='title-news') # list

    se = set()
    for i in s:
        se.add(i.find('a')['href'])
    return se

In [ ]:
def scrape_information(html_text): # scrape information of a news
    title = html_text.find('h1', class_='title-detail mt20').text.strip()
    date = html_text.find('span', class_='date').text.strip()
    
    # get content
    s = ''
    s += html_text.find('p',class_='description').text
    s += '\n'
    l = []
    tmp = html_text.find_all('p', class_='Normal')
    for i in tmp:
        l.append(i.text)
    
    author = l.pop(-1)
    content = s + '\n'.join(l)
    
    return {
        'title': title,
        'date': date,
        'author': author,
        'content': content
    }
    

In [16]:
def send_child_request(se): # loop thru a set of href attributes, send request and extract information
    res = []
    count = 0
    for link in se:
        print(f"Scraping: {link}")

        r = requests.get(link)
        
        if r.status_code == 200:
            print(f"Request sent successfully for: {link}")
            html_text = BeautifulSoup(r.text, 'html.parser')
            data = scrape_information(html_text)
            data['url'] = link
            res.append(data)
            count +=1 
        
        # scrape the first 15 news
        if count==15:
            break
        
    return res
        

In [17]:
def save_data(data, path='data/vn_express_bs4.csv'):
    fieldnames = data[0].keys()
    with open(path, mode='w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)

In [18]:
status_code, html_text = send_requests(link)
se = get_links(html_text)
res = send_child_request(se)

save_data(res)

Scraping: https://vnexpress.net/dung-dau-gia-san-van-dong-long-an-5113821.html
Request sent successfully for: https://vnexpress.net/dung-dau-gia-san-van-dong-long-an-5113821.html
Scraping: https://vnexpress.net/ha-noi-thu-nghiem-xe-buyt-tu-hanh-robotaxi-5114096.html
Request sent successfully for: https://vnexpress.net/ha-noi-thu-nghiem-xe-buyt-tu-hanh-robotaxi-5114096.html
Scraping: https://vnexpress.net/thu-tuong-giao-duc-phai-thoat-khoi-ap-luc-diem-so-5114118.html
Request sent successfully for: https://vnexpress.net/thu-tuong-giao-duc-phai-thoat-khoi-ap-luc-diem-so-5114118.html
Scraping: https://vnexpress.net/xe-cuu-thuong-tong-do-tuong-rao-benh-vien-5114740.html
Request sent successfully for: https://vnexpress.net/xe-cuu-thuong-tong-do-tuong-rao-benh-vien-5114740.html
Scraping: https://vnexpress.net/chay-xuong-nhua-gan-1-000-m2-o-ha-noi-5114487.html
Request sent successfully for: https://vnexpress.net/chay-xuong-nhua-gan-1-000-m2-o-ha-noi-5114487.html
Scraping: https://vnexpress.net

In [20]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
df = pd.read_csv('data/vn_express_bs4.csv')

df.head()


,title,date,author,content,url
0,Dừng đấu giá sân vận động Long An,"Thứ sáu, 28/8/2026, 00:00 (GMT+7)",Hoàng Nam,"Sân vận động từng là ""chảo lửa"" ở miền Tây sẽ không được đưa ra đấu giá hơn 1.800 tỷ đồng, thay vào đó nghiên cứu phương án sử dụng mới trong quy hoạch đô thị Tân An.\nNgày 27/8, UBND tỉnh Tây Ninh cho biết đã dừng chủ trương đấu giá sân vận động Long An do không còn phù hợp với tình hình thực tế và yêu cầu quản lý, sử dụng tài sản công.\nThay vì đấu giá, tỉnh sẽ nghiên cứu phương án quản lý, sử dụng tổng thể khu đất trong quá trình xây dựng quy hoạch chung đô thị Tân An, hướng đến cải thiện môi trường sống cho người dân.\nTỉnh cũng nghiên cứu vị trí xây dựng sân vận động mới, bảo đảm đồng bộ với hạ tầng xã hội và quy hoạch đô thị. Công trình mới được định hướng đủ điều kiện tổ chức các giải đấu cấp quốc gia và quốc tế.\nNằm ở trung tâm đô thị, cách TP HCM khoảng 50 km, sân vận động Long An xây cách đây 42 năm, có khuôn viên rộng gần 75.000 m2, sức chứa hơn 30.000 chỗ, từng là một trong những ""chảo lửa"" bóng đá miền Tây. Đây là sân nhà của CLB Đồng Tâm Long An trong giai đoạn đội bóng giành hai chức vô địch V-League liên tiếp năm 2005 và 2006 dưới thời HLV Calisto.\nTuy nhiên, sau nhiều năm sử dụng, công trình xuống cấp nghiêm trọng. Bốn khán đài hiện chỉ còn đáp ứng khoảng 10.000 chỗ ngồi. Đơn vị quản lý nhiều lần sửa chữa một số hạng mục để phục vụ thi đấu.\nNăm 2018, UBND tỉnh Long An quyết định bán tài sản trên đất và chuyển nhượng quyền sử dụng đất các công trình thể thao để lấy kinh phí xây dựng trung tâm thể thao mới cách sân cũ khoảng 5 km, tại xã Lợi Bình Nhơn.\nQuá trình thực hiện gặp nhiều khó khăn, trong đó có việc xác định giá do quy hoạch sử dụng đất chưa được điều chỉnh, thiếu quy hoạch chi tiết và chưa có nơi bố trí để di dời chỗ ở, tập luyện của các vận động viên.\nTháng 8/2023, UBND tỉnh quyết định chuyển giao cơ sở nhà, đất cho Trung tâm Phát triển quỹ đất thuộc Sở Tài nguyên và Môi trường để thực hiện đấu giá quyền sử dụng đất. Tuy nhiên, đến tháng 5/2026, Thường trực Tỉnh ủy Tây Ninh thống nhất tạm dừng đấu giá.\nDo cơ sở vật chất ăn ở, tập luyện không đảm bảo, các vận động viên tại Trung tâm Huấn luyện và Thi đấu thể thao Tây Ninh hiện đã được di dời đến cơ sở 2 tại xã Châu Thành.",https://vnexpress.net/dung-dau-gia-san-van-dong-long-an-5113821.html
1,"Hà Nội thử nghiệm xe buýt tự hành, robotaxi","Thứ năm, 27/8/2026, 14:04 (GMT+7)",Võ Hải,"Hà Nội cho phép thử nghiệm có kiểm soát xe buýt tự hành, robotaxi, robot dọn vệ sinh và giao hàng trên một số tuyến đường tại Hòa Lạc.\nTheo quyết định của UBND TP Hà Nội, chương trình kéo dài 24 tháng tính từ thời điểm thu thập dữ liệu đầu tiên trong khu thử nghiệm. Các phương tiện chỉ hoạt động trên tuyến cố định đã được lập bản đồ độ chính xác cao (HD Map), kết nối hệ thống giám sát và đáp ứng các yêu cầu an toàn trước khi vận hành.\nKhu vực thử nghiệm gồm Khu Công nghệ cao Hòa Lạc và cơ sở Hòa Lạc của Đại học Quốc gia Hà Nội (cách trung tâm thành phố 30 km về phía tây). Ban Quản lý các Khu công nghệ cao và Khu công nghiệp thành phố chịu trách nhiệm hướng dẫn, giám sát và kiểm soát quá trình thực hiện.\nDự án nhằm đánh giá độ an toàn, tin cậy và khả năng vận hành của phương tiện tự hành trong điều kiện giao thông thực tế. Thành phố cũng sẽ đánh giá khả năng ứng dụng vào vận tải công cộng, trải nghiệm của người dùng và mức độ kết nối với hệ thống giao thông hiện có.\nDữ liệu thu được trong quá trình thử nghiệm sẽ phục vụ nghiên cứu, xây dựng tiêu chuẩn, quy chuẩn về phương tiện tự hành tại Việt Nam.\nXe tự hành sử dụng LiDAR, camera, bản đồ HD Map, GPS độ chính xác cao và các thuật toán trí tuệ nhân tạo. Doanh nghiệp tham gia cho biết đã tự phát triển phần mềm tự hành, nền tảng quản lý đội xe và hợp tác thiết kế phần cứng.\nTốc độ tối đa dự kiến 40 km/h, thời gian hoạt động từ 9h đến 16h30. Trong 12 tháng thử nghiệm công khai, xe buýt tự hành dự kiến phục vụ 40.000-50.000 lượt khách, robotaxi 20.000-35.000 lượt. Mỗi chuyến xe buýt chở tối đa 8 hành khá